In [1]:
import pandas as pd
import numpy as np
import os

In [3]:
secondary_df = pd.read_csv("../data/processed/secondary_sales_cleaned.csv")

underconstruction_df = pd.read_csv("../data/processed/under_construction_cleaned.csv")

In [4]:
df = secondary_df.copy()

In [5]:
print(df.shape)

df.head()

(50000, 40)


,id,date_listed,locality,region,tier,lat,lon,property_type,bedrooms,carpet_area_sqft,...,price_usd,home_loan_rate_at_listing,listing_year,listing_month,property_age,calculated_price_per_sqft,luxury_score,connectivity_score,investment_score,investment_category
0,S000001,2024-11-23,Breach Candy,South Mumbai,luxury,18.96953,72.81853,3BHK,3,1048,...,740322,9.20,2024,11,13,58868.320611,5,81,45.694809,Medium
1,S000002,2023-07-05,Breach Candy,South Mumbai,luxury,18.97230,72.81017,2BHK,2,729,...,481523,9.20,2023,7,35,55043.895748,3,44,29.532721,Medium
2,S000003,2022-01-22,Malad East,Western,mid,19.18986,72.85940,2BHK,2,705,...,125398,7.10,2022,1,10,14822.695035,2,81,36.058294,Medium
3,S000004,2020-11-18,Mulund West,Central,mid,19.15854,72.93798,1BHK,1,424,...,65036,6.70,2020,11,12,12783.018868,0,10,6.740702,Low
4,S000005,2025-01-11,Matunga,Central,premium,19.02358,72.85690,1BHK,1,392,...,140553,8.95,2025,1,2,29880.102041,0,82,38.857375,Medium


In [6]:
os.makedirs("../dashboard_data", exist_ok=True)

# Creating Business KPIs

In [7]:
kpi_data = {
    "Metric": [
        "Average Property Price",
        "Average Price Per Sqft",
        "Average Carpet Area",
        "Average Metro Distance",
        "Total Properties",
        "Average Luxury Score"
    ],

    "Value": [
        df["price_inr"].mean(),
        df["price_per_sqft_carpet_inr"].mean(),
        df["carpet_area_sqft"].mean(),
        df["metro_distance_min"].mean(),
        len(df),
        df["luxury_score"].mean()
    ]
}

kpi_df = pd.DataFrame(kpi_data)

kpi_df

,Metric,Value
0,Average Property Price,2.312204e+07
1,Average Price Per Sqft,2.400652e+04
2,Average Carpet Area,8.966935e+02
3,Average Metro Distance,3.246326e+01
4,Total Properties,5.000000e+04
5,Average Luxury Score,1.866720e+00


In [8]:
kpi_df.to_csv("../dashboard_data/kpi_data.csv", index=False)

In [9]:
locality_analysis = df.groupby("locality").agg({
    "price_inr": "mean",
    "price_per_sqft_carpet_inr": "mean",
    "carpet_area_sqft": "mean",
    "metro_distance_min": "mean",
    "luxury_score": "mean"
}).reset_index()

In [10]:
locality_analysis.columns = [
    "Locality",
    "Avg_Price",
    "Avg_Price_Per_Sqft",
    "Avg_Carpet_Area",
    "Avg_Metro_Distance",
    "Avg_Luxury_Score"
]

In [11]:
locality_analysis = locality_analysis.sort_values(
    by="Avg_Price",
    ascending=False
)

In [12]:
locality_analysis.to_csv(
    "../dashboard_data/locality_analysis.csv",
    index=False
)

In [13]:
locality_analysis.head()

,Locality,Avg_Price,Avg_Price_Per_Sqft,Avg_Carpet_Area,Avg_Metro_Distance,Avg_Luxury_Score
42,Malabar Hill,8.065866e+07,69257.279933,1167.239460,28.252951,4.811130
13,Breach Candy,7.866230e+07,65879.609508,1193.886248,23.062818,4.932088
17,Cuffe Parade,7.776892e+07,66784.924959,1167.507341,8.265905,4.853181
82,Walkeshwar,7.417704e+07,64380.944079,1150.472039,29.463816,4.865132
4,BKC (Bandra-Kurla),7.280095e+07,61757.513289,1171.598007,8.892027,4.860465


# Investment Score Engineering

In [14]:
# Aprreciation Score
df["appreciation_score"] = (
    df["price_per_sqft_carpet_inr"] /
    df["price_per_sqft_carpet_inr"].max()
) * 100


# Affordabilty Score
df["affordability_score"] = (
    1 - (df["price_inr"] / df["price_inr"].max())
) * 100


# Demand Score
locality_frequency = df["locality"].value_counts()
df["demand_score"] = df["locality"].map(locality_frequency)


# Normalizing it
df["demand_score"] = (
    df["demand_score"] /
    df["demand_score"].max()
) * 100


# Rental Yield Score
df["rental_yield_score"] = (
    df["price_per_sqft_carpet_inr"] /
    df["price_per_sqft_carpet_inr"].max()
) * 100


# Final Investment Svore
df["investment_score"] = (
    0.4 * df["appreciation_score"] +
    0.3 * df["rental_yield_score"] +
    0.2 * df["demand_score"] +
    0.1 * df["affordability_score"]
)


# Creating Investment category
def investment_category(score):

    if score >= 70:
        return "High Potential"

    elif score >= 45:
        return "Medium Potential"

    else:
        return "Low Potential"
    

df["investment_category"] = df["investment_score"].apply(
    investment_category
)




In [15]:
# Checking The Results

df[[
    "locality",
    "investment_score",
    "investment_category"
]].head()

,locality,investment_score,investment_category
0,Breach Candy,62.864043,Medium Potential
1,Breach Candy,61.103446,Medium Potential
2,Malad East,36.898077,Low Potential
3,Mulund West,35.567596,Low Potential
4,Matunga,46.018776,Medium Potential


In [16]:
# Saving the investment Datset
df.to_csv(
    "../dashboard_data/investment_analysis.csv",
    index=False
)

# HOTSPOT ANALYSIS

In [17]:
# Top Investment Hotspots
hotspots = df.groupby("locality").agg({
    "investment_score": "mean",
    "price_inr": "mean",
    "luxury_score": "mean"
}).reset_index()

In [18]:
# Sort Hotspots
hotspots = hotspots.sort_values(
    by="investment_score",
    ascending=False
)

In [19]:
# Save Hotspots
hotspots.to_csv(
    "../dashboard_data/hotspot_analysis.csv",
    index=False
)

# GEO ANALYSIS

In [20]:
# Creating Geo Dataset
geo_data = df[[
    "locality",
    "lat",
    "lon",
    "price_inr",
    "investment_score",
    "luxury_score"
]]

geo_data.to_csv(
    "../dashboard_data/geo_analysis.csv",
    index=False
)

# Checking

In [21]:
os.listdir("../dashboard_data")

['geo_analysis.csv',
 'hotspot_analysis.csv',
 'investment_analysis.csv',
 'kpi_data.csv',
 'locality_analysis.csv']